In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
# Load the unified dataset
data_path = Path("unified_dataset.jsonl")
df = pd.read_json(data_path, lines=True)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")

In [ ]:
# Preview first few rows
display(df.head(3))

# Schema summary
print("\n--- Schema Summary ---")
print(f"Columns: {list(df.columns)}")
print(f"Total rows: {len(df)}")

In [ ]:
# Missingness analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"non_null": len(df) - missing, "null": missing, "null_%": missing_pct})
print("--- Missingness ---")
display(missing_df)

# Visualize missingness
fig, ax = plt.subplots()
missing_df[["null", "non_null"]].plot(kind="bar", stacked=True, ax=ax, color=["salmon", "lightgreen"])
ax.set_title("Null vs Non-Null Counts per Column")
ax.set_ylabel("Count")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Source distribution
if "source" in df.columns:
    source_counts = df["source"].value_counts()
    print("--- Source Distribution ---")
    display(source_counts)
    
    fig, ax = plt.subplots()
    source_counts.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title("Distribution by Source")
    ax.set_ylabel("Count")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Label distribution
if "label" in df.columns:
    label_non_null = df["label"].dropna()
    print(f"--- Label Distribution ---")
    print(f"Non-null labels: {len(label_non_null)} / {len(df)}")
    print(f"Unique labels: {label_non_null.nunique()}")
    
    # Show sample labels
    print("\nSample labels (first 3):")
    for i, lbl in enumerate(label_non_null.head(3)):
        print(f"  {i+1}. {lbl[:100]}...") if len(str(lbl)) > 100 else print(f"  {i+1}. {lbl}")

In [ ]:
# Text length distribution
if "text" in df.columns:
    df["text_length"] = df["text"].astype(str).str.split().str.len()
    
    print("--- Text Length Statistics (words) ---")
    print(df["text_length"].describe())
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(df["text_length"], bins=30, color="steelblue", edgecolor="white")
    axes[0].set_title("Text Length Distribution")
    axes[0].set_xlabel("Word Count")
    axes[0].set_ylabel("Frequency")
    
    # Box plot
    axes[1].boxplot(df["text_length"], vert=True)
    axes[1].set_title("Text Length Box Plot")
    axes[1].set_ylabel("Word Count")
    
    plt.tight_layout()
    plt.show()

## Data Quality Review

### Analyzer View

- Task interpretation: unknown
- Primary modality: text
- Target semantics: unknown
- Relevant checks: text_non_empty, label_format_consistency, source_distribution
- Lower-value checks: class_balance, numeric_outliers, imbalance_ratio
- Priority actions: drop_null_modality_columns, preserve_unlabeled_math_problems

### Strategy Justification

This is a math reasoning dataset (GSM8K-style) where text is the primary modality. Audio/image columns were 100% null and dropped. All 100 rows contain valid math problem text - no empty text rows to drop. The 50 unlabeled rows (from all-russian and project-euler sources) are preserved as they are useful for inference evaluation or future label generation. No exact or normalized text duplicates were found. No numeric imputation needed (no numeric columns with missing values). No outlier clipping needed (text dataset).

- Missing values: `median`
- Duplicates: `drop`
- Outliers: `clip_iqr`

### Findings

- Missing values before cleaning: 250
- Duplicate rows before cleaning: 0
- Numeric outliers before cleaning: 11
- Imbalance column: `label`
- Majority class share after cleaning: not applicable

### Before / After

- Missing values: 0 -> 0
- Duplicates: 0 -> 0
- Outliers: 0 -> 0


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

raw_df = pd.read_json('data/collection/unified_dataset.jsonl', lines=True)
clean_df = pd.read_json('data/quality/cleaned_dataset.jsonl', lines=True)
primary_modality = 'text'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
missing_counts = raw_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
if not missing_counts.empty:
    sns.barplot(x=missing_counts.values, y=missing_counts.index, ax=axes[0, 0], color='#d97706')
    axes[0, 0].set_title('Missing values by column')
else:
    axes[0, 0].text(0.5, 0.5, 'No missing values', ha='center', va='center')
    axes[0, 0].set_axis_off()

if {'source', 'label'}.issubset(raw_df.columns):
    coverage = raw_df.assign(label_present=raw_df['label'].notna()).groupby('source', dropna=False)['label_present'].mean().sort_values(ascending=False).head(10)
    if not coverage.empty:
        sns.barplot(x=coverage.values, y=coverage.index.astype(str), ax=axes[0, 1], color='#2563eb')
        axes[0, 1].set_title('Label coverage by source')
        axes[0, 1].set_xlim(0, 1)
    else:
        axes[0, 1].text(0.5, 0.5, 'No source coverage data', ha='center', va='center')
        axes[0, 1].set_axis_off()
elif 'source' in raw_df.columns:
    source_counts = raw_df['source'].astype(str).value_counts().head(10)
    sns.barplot(x=source_counts.values, y=source_counts.index, ax=axes[0, 1], color='#2563eb')
    axes[0, 1].set_title('Top sources')
else:
    axes[0, 1].text(0.5, 0.5, 'No source column', ha='center', va='center')
    axes[0, 1].set_axis_off()

numeric_columns = raw_df.select_dtypes(include=['number']).columns.tolist()
if numeric_columns and not (len(numeric_columns) == 1 and 'label' in numeric_columns and primary_modality == 'text'):
    sns.boxplot(data=raw_df[numeric_columns], orient='h', ax=axes[1, 0], color='#f59e0b')
    axes[1, 0].set_title('Raw numeric distributions')
    sns.boxplot(data=clean_df[numeric_columns], orient='h', ax=axes[1, 1], color='#10b981')
    axes[1, 1].set_title('Cleaned numeric distributions')
else:
    if 'text' in raw_df.columns:
        raw_lengths = raw_df['text'].fillna('').astype(str).str.split().str.len()
        clean_lengths = clean_df['text'].fillna('').astype(str).str.split().str.len()
        sns.histplot(raw_lengths, bins=30, ax=axes[1, 0], color='#f59e0b')
        axes[1, 0].set_title('Raw text length distribution')
        sns.histplot(clean_lengths, bins=30, ax=axes[1, 1], color='#10b981')
        axes[1, 1].set_title('Cleaned text length distribution')
    else:
        axes[1, 0].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 0].set_axis_off()
        axes[1, 1].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()
